<a href="https://colab.research.google.com/github/irenezvirkovska-hue/infectious-disease-sql-analysis/blob/main/infectious_disease_sql_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Infectious Disease Data Analysis with PostgreSQL

## Load the data

Upload the CSV to Colab and inspect the table dimensions, column types, and missing values.

In [1]:
from pathlib import Path
from google.colab import files
import pandas as pd

csv_path = Path("infectious_cases.csv")
if not csv_path.exists():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload one CSV file: infectious_cases.csv.")
    csv_path = Path(next(iter(uploaded)))
    if csv_path.suffix.lower() != ".csv":
        raise ValueError("A CSV file is required.")

df = pd.read_csv(csv_path)
required_columns = {
    "Entity", "Code", "Year", "Number_yaws", "polio_cases",
    "cases_guinea_worm", "Number_rabies", "Number_malaria",
    "Number_hiv", "Number_tuberculosis", "Number_smallpox",
    "Number_cholera_cases",
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing CSV columns: {sorted(missing_columns)}")
if df.empty:
    raise ValueError("The CSV file contains no data.")
if df[["Entity", "Year"]].isna().any().any():
    raise ValueError("Entity and year must not be missing.")
years = pd.to_numeric(df["Year"], errors="coerce")
if years.isna().any() or (years % 1 != 0).any() or (years < 1).any() or (years > 9999).any():
    raise ValueError("Year must contain integers from 1 to 9999.")
df["Year"] = years.astype(int)
print("Dataset shape:", df.shape)
display(df.head())
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum()}))


Saving infectious_cases.csv to infectious_cases.csv
Dataset shape: (10521, 12)


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,NaN,6160.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1981,NaN,5859.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1982,NaN,9730.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1983,NaN,13937.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1984,NaN,3864.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


,dtype,missing
Entity,object,0
Code,object,1128
Year,int64,0
Number_yaws,float64,10447
polio_cases,float64,2911
cases_guinea_worm,float64,2093
Number_rabies,float64,3741
Number_malaria,float64,3741
Number_hiv,float64,3741
Number_tuberculosis,float64,3741


## Connect to PostgreSQL

Install the dependencies and start PostgreSQL in the Colab runtime.

In [2]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 108.2 MB/s eta 0:00:00


In [3]:
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/fp_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## Import into the database

Create the `pandemic` schema, import the data into `infectious_cases`, and check the row count. Rerunning the import replaces the source table.

In [4]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS pandemic"))
    conn.execute(text("SET search_path TO pandemic"))

print("Schema pandemic created")

Schema pandemic created


In [5]:
df.to_sql(
    name="infectious_cases",
    con=engine,
    schema="pandemic",
    if_exists="replace",
    index=False,
    chunksize=1000
)

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM pandemic.infectious_cases")
    ).scalar_one()

print("Rows imported:", count)

Rows imported: 10521


In [6]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT *
        FROM pandemic.infectious_cases
        LIMIT 5
    """))

    for row in result:
        print(row)

('Afghanistan', 'AFG', 1980, None, 6160.0, 0.0, None, None, None, None, None, None)
('Afghanistan', 'AFG', 1981, None, 5859.0, 0.0, None, None, None, None, None, None)
('Afghanistan', 'AFG', 1982, None, 9730.0, 0.0, None, None, None, None, None, None)
('Afghanistan', 'AFG', 1983, None, 13937.0, 0.0, None, None, None, None, None, None)
('Afghanistan', 'AFG', 1984, None, 3864.0, 0.0, None, None, None, None, None, None)


## Normalize the data

Count the distinct `Entity`–`Code` pairs and extract them into the `entities` lookup table.

In [7]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT ("Entity", "Code")) AS unique_pairs
        FROM pandemic.infectious_cases
    """))

    print(result.fetchone())

(10521, 245)


In [8]:
create_entities_sql = """
DROP TABLE IF EXISTS pandemic.infectious_cases_normalized;
DROP TABLE IF EXISTS pandemic.entities;

CREATE TABLE pandemic.entities (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity TEXT NOT NULL,
    code TEXT,
    UNIQUE (entity, code)
);

INSERT INTO pandemic.entities (entity, code)
SELECT DISTINCT
    "Entity",
    "Code"
FROM pandemic.infectious_cases;
"""

with engine.begin() as conn:
    conn.execute(text(create_entities_sql))

In [9]:
with engine.connect() as conn:
    entities_df = pd.read_sql(
        """
        SELECT *
        FROM pandemic.entities
        ORDER BY id
        LIMIT 10
        """,
        con=conn
    )

display(entities_df)

,id,entity,code
0,1,Morocco,MAR
1,2,Russia,RUS
2,3,Democratic Republic of Congo,COD
3,4,Trinidad and Tobago,TTO
4,5,Antigua and Barbuda,ATG
5,6,Libya,LBY
6,7,Liberia,LBR
7,8,Greece,GRC
8,9,Cuba,CUB
9,10,Nicaragua,NIC


Store annual indicators in `infectious_cases_normalized`, referencing `entities`. Use `IS NOT DISTINCT FROM` to match missing codes correctly.

In [10]:
create_normalized_sql = """
CREATE TABLE pandemic.infectious_cases_normalized (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity_id INTEGER NOT NULL,
    year INTEGER NOT NULL,
    number_yaws DOUBLE PRECISION,
    polio_cases DOUBLE PRECISION,
    cases_guinea_worm DOUBLE PRECISION,
    number_rabies DOUBLE PRECISION,
    number_malaria DOUBLE PRECISION,
    number_hiv DOUBLE PRECISION,
    number_tuberculosis DOUBLE PRECISION,
    number_smallpox DOUBLE PRECISION,
    number_cholera_cases DOUBLE PRECISION,

    FOREIGN KEY (entity_id)
        REFERENCES pandemic.entities(id)
);

INSERT INTO pandemic.infectious_cases_normalized (
    entity_id,
    year,
    number_yaws,
    polio_cases,
    cases_guinea_worm,
    number_rabies,
    number_malaria,
    number_hiv,
    number_tuberculosis,
    number_smallpox,
    number_cholera_cases
)
SELECT
    e.id,
    ic."Year",
    ic."Number_yaws",
    ic."polio_cases",
    ic."cases_guinea_worm",
    ic."Number_rabies",
    ic."Number_malaria",
    ic."Number_hiv",
    ic."Number_tuberculosis",
    ic."Number_smallpox",
    ic."Number_cholera_cases"
FROM pandemic.infectious_cases AS ic
JOIN pandemic.entities AS e
    ON ic."Entity" = e.entity
   AND ic."Code" IS NOT DISTINCT FROM e.code;
"""

with engine.begin() as conn:
    conn.execute(text(create_normalized_sql))

print("Normalized rows:")
with engine.connect() as conn:
    print(conn.execute(text("""
        SELECT COUNT(*)
        FROM pandemic.infectious_cases_normalized
    """)).scalar_one())

    display(pd.read_sql("""
        SELECT *
        FROM pandemic.infectious_cases_normalized
        ORDER BY id
        LIMIT 5
    """, con=conn))

Normalized rows:
10521


,id,entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,1,189,2021,None,None,0.0,NaN,NaN,NaN,NaN,None,4755.0
1,2,189,1996,None,None,0.0,3.281437,665807.44,152.26442,22701.540,None,NaN
2,3,189,1994,None,None,0.0,2.989987,818779.50,133.06308,21658.172,None,38735.0
3,4,189,1993,None,None,0.0,2.853117,963817.10,127.26330,21195.934,None,37046.0
4,5,189,1992,None,None,0.0,2.618041,1081609.60,119.93340,19786.672,None,NaN


## Validate relationships

Compare row counts before and after normalization and check for records without a matching entity in the lookup table.

In [11]:
with engine.connect() as conn:
    checks = pd.read_sql(text("""
        SELECT
            (SELECT COUNT(*) FROM pandemic.infectious_cases) AS original_rows,
            (SELECT COUNT(*) FROM pandemic.infectious_cases_normalized) AS normalized_rows,
            (SELECT COUNT(*) FROM pandemic.entities) AS entities_count,
            (SELECT COUNT(*)
             FROM pandemic.infectious_cases_normalized n
             LEFT JOIN pandemic.entities e ON e.id = n.entity_id
             WHERE e.id IS NULL) AS orphan_rows,
            (SELECT COUNT(*) FROM (
                SELECT entity, code FROM pandemic.entities
                GROUP BY entity, code HAVING COUNT(*) > 1
             ) duplicates) AS duplicate_entities
    """), con=conn)

row = checks.iloc[0]
assert row["original_rows"] == row["normalized_rows"], "The row count changed after normalization."
assert row["orphan_rows"] == 0, "Some records have no matching entity."
assert row["duplicate_entities"] == 0, "The lookup table contains duplicate entities."
display(checks)
print("Normalization checks passed.")

,original_rows,normalized_rows,entities_count,orphan_rows,duplicate_entities
0,10521,10521,245,0,0


Normalization checks passed.


## Rabies indicators

Calculate the mean, minimum, maximum, and sum for each entity and return the top 10 by mean. The dataset includes countries and aggregate groups; missing values are not replaced with zeros.

In [12]:
rabies_sql = text("""
    SELECT
        e.entity,
        e.code,
        COUNT(DISTINCT n.year) AS years_with_data,
        MIN(n.year) AS first_year,
        MAX(n.year) AS last_year,
        AVG(n.number_rabies) AS average_rabies,
        MIN(n.number_rabies) AS min_rabies,
        MAX(n.number_rabies) AS max_rabies,
        SUM(n.number_rabies) AS total_rabies
    FROM pandemic.infectious_cases_normalized AS n
    JOIN pandemic.entities AS e ON e.id = n.entity_id
    WHERE n.number_rabies IS NOT NULL
    GROUP BY e.id, e.entity, e.code
    ORDER BY average_rabies DESC, e.entity, e.code NULLS LAST
    LIMIT 10
""")
with engine.connect() as conn:
    rabies_df = pd.read_sql(rabies_sql, con=conn)
display(rabies_df)

,entity,code,years_with_data,first_year,last_year,average_rabies,min_rabies,max_rabies,total_rabies
0,World,OWID_WRL,30,1990,2019,20192.370367,14075.5080,24744.6580,605771.1110
1,Lower Middle Income (WB),None,30,1990,2019,15193.959383,10202.5280,19182.7950,455818.7815
2,South Asia (WB),None,30,1990,2019,11729.888753,7271.2754,15361.8780,351896.6626
3,South-East Asia Region (WHO),None,30,1990,2019,11424.326893,6806.0127,15641.9580,342729.8068
4,G20,None,30,1990,2019,10189.046003,6339.0796,13164.8810,305671.3801
5,India,IND,30,1990,2019,8599.173280,5425.8726,11121.1420,257975.1984
6,Sub-Saharan Africa (WB),None,30,1990,2019,4748.268090,4356.8184,5017.9770,142448.0427
7,African Region (WHO),None,30,1990,2019,4649.702377,4283.0730,4917.7840,139491.0713
8,Low Income (WB),None,30,1990,2019,3568.105450,3014.6170,3873.5330,107043.1635
9,East Asia & Pacific (WB),None,30,1990,2019,3427.461133,2009.9395,4591.0107,102823.8340


## Date calculations

Construct January 1 for each observation year and calculate the number of complete years up to the current date.

In [13]:
year_diff_sql = text("""
    SELECT e.entity, e.code, n.year,
        MAKE_DATE(n.year, 1, 1) AS year_start,
        CURRENT_DATE AS current_date,
        EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(n.year, 1, 1)))::INTEGER AS year_difference
    FROM pandemic.infectious_cases_normalized n
    JOIN pandemic.entities e ON e.id = n.entity_id
    ORDER BY n.year, e.entity, e.code NULLS LAST, n.id
    LIMIT 10
""")
with engine.connect() as conn:
    year_diff_df = pd.read_sql(year_diff_sql, con=conn)
display(year_diff_df)

,entity,code,year,year_start,current_date,year_difference
0,World,OWID_WRL,1920,1920-01-01,2026-10-27,106
1,World,OWID_WRL,1921,1921-01-01,2026-10-27,105
2,World,OWID_WRL,1922,1922-01-01,2026-10-27,104
3,World,OWID_WRL,1923,1923-01-01,2026-10-27,103
4,World,OWID_WRL,1924,1924-01-01,2026-10-27,102
5,World,OWID_WRL,1925,1925-01-01,2026-10-27,101
6,World,OWID_WRL,1926,1926-01-01,2026-10-27,100
7,World,OWID_WRL,1927,1927-01-01,2026-10-27,99
8,World,OWID_WRL,1928,1928-01-01,2026-10-27,98
9,World,OWID_WRL,1929,1929-01-01,2026-10-27,97


## Year difference function

Move the calculation into `year_difference()` and test it with a single value and the table data.

In [14]:
create_year_function_sql = """
CREATE OR REPLACE FUNCTION pandemic.year_difference(input_year INTEGER)
RETURNS INTEGER
LANGUAGE SQL
AS $$
    SELECT EXTRACT(
        YEAR FROM AGE(
            CURRENT_DATE,
            MAKE_DATE(input_year, 1, 1)
        )
    )::INTEGER;
$$;
"""

with engine.begin() as conn:
    conn.execute(text(create_year_function_sql))

print("Function year_difference created")

Function year_difference created


In [15]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            1996 AS input_year,
            pandemic.year_difference(1996) AS year_difference
    """))

    print(result.fetchone())

(1996, 30)


In [16]:
with engine.connect() as conn:
    result = pd.read_sql(text("""
        SELECT e.entity, e.code, n.year,
            pandemic.year_difference(n.year) AS year_difference
        FROM pandemic.infectious_cases_normalized n
        JOIN pandemic.entities e ON e.id = n.entity_id
        ORDER BY n.year, e.entity, e.code NULLS LAST, n.id
        LIMIT 10
    """), con=conn)
display(result)

,entity,code,year,year_difference
0,World,OWID_WRL,1920,106
1,World,OWID_WRL,1921,105
2,World,OWID_WRL,1922,104
3,World,OWID_WRL,1923,103
4,World,OWID_WRL,1924,102
5,World,OWID_WRL,1925,101
6,World,OWID_WRL,1926,100
7,World,OWID_WRL,1927,99
8,World,OWID_WRL,1928,98
9,World,OWID_WRL,1929,97


## Average per period

The `average_cases()` function divides an annual indicator by a specified number of periods. It returns `NULL` for a zero divisor or missing arguments.

In [17]:
create_average_function_sql = """
CREATE OR REPLACE FUNCTION pandemic.average_cases(
    cases_value DOUBLE PRECISION, divisor DOUBLE PRECISION
)
RETURNS DOUBLE PRECISION
LANGUAGE SQL
IMMUTABLE
AS $$ SELECT cases_value / NULLIF(divisor, 0); $$;
"""
with engine.begin() as conn:
    conn.execute(text(create_average_function_sql))
with engine.connect() as conn:
    assert conn.execute(text("SELECT pandemic.average_cases(120, 12)")).scalar_one() == 10
    assert conn.execute(text("SELECT pandemic.average_cases(120, 0)")).scalar_one() is None
    assert conn.execute(text("SELECT pandemic.average_cases(NULL, 12)")).scalar_one() is None
    assert conn.execute(text("SELECT pandemic.average_cases(120, NULL)")).scalar_one() is None
print("Function created; checks passed.")

Function created; checks passed.


Demonstrate division by 12, 4, and 2. These are period averages, not observed monthly or quarterly values.

In [18]:
with engine.connect() as conn:
    result = pd.read_sql(text("""
        SELECT e.entity, e.code, n.year, n.number_rabies,
            pandemic.average_cases(n.number_rabies, 12) AS monthly_avg,
            pandemic.average_cases(n.number_rabies, 4) AS quarterly_avg,
            pandemic.average_cases(n.number_rabies, 2) AS half_year_avg
        FROM pandemic.infectious_cases_normalized n
        JOIN pandemic.entities e ON e.id = n.entity_id
        WHERE n.number_rabies IS NOT NULL
        ORDER BY e.entity, e.code NULLS LAST, n.year, n.id
        LIMIT 10
    """), con=conn)
display(result)

,entity,code,year,number_rabies,monthly_avg,quarterly_avg,half_year_avg
0,Afghanistan,AFG,1990,2.145772,0.178814,0.536443,1.072886
1,Afghanistan,AFG,1991,2.320975,0.193415,0.580244,1.160487
2,Afghanistan,AFG,1992,2.618041,0.218170,0.654510,1.309020
3,Afghanistan,AFG,1993,2.853117,0.237760,0.713279,1.426559
4,Afghanistan,AFG,1994,2.989987,0.249166,0.747497,1.494994
5,Afghanistan,AFG,1995,3.126361,0.260530,0.781590,1.563181
6,Afghanistan,AFG,1996,3.281437,0.273453,0.820359,1.640718
7,Afghanistan,AFG,1997,3.446122,0.287177,0.861530,1.723061
8,Afghanistan,AFG,1998,3.557025,0.296419,0.889256,1.778512
9,Afghanistan,AFG,1999,3.607037,0.300586,0.901759,1.803519


In [19]:
with engine.connect() as conn:
    latest_results = pd.read_sql(text("""
        SELECT e.entity, e.code, n.year,
            pandemic.year_difference(n.year) AS year_difference,
            n.number_rabies,
            pandemic.average_cases(n.number_rabies, 12) AS monthly_avg,
            pandemic.average_cases(n.number_rabies, 4) AS quarterly_avg,
            pandemic.average_cases(n.number_rabies, 2) AS half_year_avg
        FROM pandemic.infectious_cases_normalized n
        JOIN pandemic.entities e ON e.id = n.entity_id
        WHERE n.number_rabies IS NOT NULL
        ORDER BY n.year DESC, e.entity, e.code NULLS LAST, n.id
        LIMIT 10
    """), con=conn)
display(latest_results)

,entity,code,year,year_difference,number_rabies,monthly_avg,quarterly_avg,half_year_avg
0,Afghanistan,AFG,2019,7,3.387213,0.282268,0.846803,1.693607
1,African Region (WHO),None,2019,7,4378.748000,364.895667,1094.687000,2189.374000
2,Albania,ALB,2019,7,0.014225,0.001185,0.003556,0.007112
3,Algeria,DZA,2019,7,1.346161,0.112180,0.336540,0.673080
4,American Samoa,ASM,2019,7,0.007448,0.000621,0.001862,0.003724
5,Andorra,AND,2019,7,0.000138,0.000012,0.000035,0.000069
6,Angola,AGO,2019,7,7.964379,0.663698,1.991095,3.982190
7,Antigua and Barbuda,ATG,2019,7,0.004845,0.000404,0.001211,0.002422
8,Argentina,ARG,2019,7,0.567866,0.047322,0.141967,0.283933
9,Armenia,ARM,2019,7,0.369019,0.030752,0.092255,0.184509


In [20]:
from IPython.display import Markdown

display(Markdown("## Changes in the indicator for Ukraine\nCompare the available annual rabies indicator values. Percentage changes are calculated only for consecutive calendar years."))
country_sql = text("""
    WITH yearly AS (
        SELECT e.entity, n.year, n.number_rabies,
            LAG(n.year) OVER (ORDER BY n.year) AS previous_year,
            LAG(n.number_rabies) OVER (ORDER BY n.year) AS previous_value
        FROM pandemic.infectious_cases_normalized n
        JOIN pandemic.entities e ON e.id = n.entity_id
        WHERE e.code = :country_code
    )
    SELECT entity, year, number_rabies,
        CASE WHEN year = previous_year + 1
             THEN 100.0 * (number_rabies - previous_value) / NULLIF(previous_value, 0)
        END AS change_pct
    FROM yearly
    ORDER BY year
""")
with engine.connect() as conn:
    country_df = pd.read_sql(country_sql, con=conn, params={"country_code": "UKR"})
assert not country_df.empty, "The dataset contains no data for UKR."
assert not country_df["year"].duplicated().any(), "There is more than one record per year."
display(country_df)
observed = country_df.dropna(subset=["number_rabies"])
summary = ""
if len(observed) >= 2:
    first, last = observed.iloc[0], observed.iloc[-1]
    summary = (
        f" For Ukraine, the first available value in {int(first['year'])} was "
        f"{first['number_rabies']:.2f}, and the last value in {int(last['year'])} was "
        f"{last['number_rabies']:.2f}. This compares the endpoints and does not establish a steady trend."
    )
display(Markdown(
    "## Conclusion\n"
    "The data was imported into PostgreSQL and separated into an entity lookup table and annual indicators. "
    "Validation confirmed that row counts and referential integrity were preserved. "
    "Comparisons of averages must account for missing values, different observation periods, "
    "and the presence of aggregate groups alongside individual countries." + summary
))

## Changes in the indicator for Ukraine
Compare the available annual rabies indicator values. Percentage changes are calculated only for consecutive calendar years.

,entity,year,number_rabies,change_pct
0,Ukraine,1980,NaN,NaN
1,Ukraine,1981,NaN,NaN
2,Ukraine,1982,NaN,NaN
3,Ukraine,1983,NaN,NaN
4,Ukraine,1984,NaN,NaN
5,Ukraine,1985,NaN,NaN
6,Ukraine,1986,NaN,NaN
7,Ukraine,1987,NaN,NaN
8,Ukraine,1988,NaN,NaN
9,Ukraine,1989,NaN,NaN


## Conclusion
The data was imported into PostgreSQL and separated into an entity lookup table and annual indicators. Validation confirmed that row counts and referential integrity were preserved. Comparisons of averages must account for missing values, different observation periods, and the presence of aggregate groups alongside individual countries. For Ukraine, the first available value in 1990 was 4.34, and the last value in 2019 was 5.22. This compares the endpoints and does not establish a steady trend.